# AraVISTA Frozen Pairwise Relation Reasoner v1.1 — Implementation-Aligned Rerun

**Status:** approved ontology/implementation correction after the first run exposed execution defects.

This rerun preserves all predeclared model identities, thresholds, reranking weights, bootstrap settings, and action priority. It corrects only implementation defects and aligns the pairwise construct with Relation Calibration Round 3.

Key construct separation:

\[
R_{ij}=g(e_i,e_j)
\]

is **query-independent pairwise evidence relation**, while

\[
S_i=h(e_i,q,t_q,E_q,R)
\]

is **query-conditioned evidence state**.

Gold answers, QCES labels, action labels, candidate answers, and relation labels are still loaded **only after inference has finished**.

In [ ]:
!pip -q install rank-bm25 sentence-transformers transformers accelerate scikit-learn scipy pandas numpy


## 1. Frozen configuration


In [ ]:
ALLOW_TEST_EVALUATION = True  # Relation kappa=0.867; Action kappa=0.85.

RUN_VERSION = "v1.1-implementation-aligned"
RELATION_GOLD_FILE = "relation_graph_v1.1_R3_aligned.csv"
CORPUS_REFERENCE_TIME = "2026-08-30"

DENSE_MODEL_ID = "intfloat/multilingual-e5-small"
NLI_MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# Frozen thresholds: unchanged from the original preregistered run.
NLI_CONTRADICTION_THRESHOLD = 0.55
NLI_ENTAILMENT_THRESHOLD = 0.55
SEMANTIC_RELATION_FLOOR = 0.40
CURRENT_CONFLICT_SUPPORT_FLOOR = 0.40

# Frozen reranking weights: unchanged.
SEMANTIC_WEIGHT = 1.00
STATE_WEIGHT = 0.85
GRAPH_WEIGHT = 0.25

STATE_BONUS = {
    "ACTIVE_SUPPORT": 1.00,
    "HISTORICAL_SUPPORT": 1.00,
    "SUPERSEDED": -1.00,
    "CO_TEMPORAL_CONFLICT": -0.80,
    "INSUFFICIENT": -0.65,
}

BOOTSTRAP_RESAMPLES = 10_000
RANDOM_SEED = 20260917

print("Frozen configuration loaded.")
print("RUN_VERSION =", RUN_VERSION)
print("ALLOW_TEST_EVALUATION =", ALLOW_TEST_EVALUATION)


## 2. Imports and reproducibility manifest


In [ ]:
import os, re, json, math, random, platform
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import scipy
from scipy.special import softmax
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
import transformers
import sentence_transformers
import sklearn

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

RUN_ENV = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "sklearn": sklearn.__version__,
    "dense_model_id": DENSE_MODEL_ID,
    "nli_model_id": NLI_MODEL_ID,
}
print(json.dumps(RUN_ENV, indent=2))


## 3. Upload leakage-safe AraVISTA-Bench v1.1 inputs

Upload only:

- `model_input_queries_v1.1.csv`
- `model_input_evidence_v1.1.csv`
- `frozen_test_manifest_v1.1.csv`

Do **not** upload gold files in this section.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
Q = pd.read_csv("model_input_queries_v1.1.csv")
E = pd.read_csv("model_input_evidence_v1.1.csv")
MANIFEST = pd.read_csv("frozen_test_manifest_v1.1.csv")

required_q = {
    "query_id","scenario_id","query_text","reference_surface",
    "normalized_reference_time","dialect","calendar_surface"
}
required_e = {"evidence_id","scenario_id","evidence_text","source_url","source_date"}

assert required_q.issubset(Q.columns)
assert required_e.issubset(E.columns)

# Leakage guard.
FORBIDDEN = {
    "gold_answer","gold_action","gold_evidence_ids","gold_evidence_id",
    "qces_state","gold_relation","candidate_answer","valid_from","valid_to"
}
assert not (FORBIDDEN & set(Q.columns)), FORBIDDEN & set(Q.columns)
assert not (FORBIDDEN & set(E.columns)), FORBIDDEN & set(E.columns)

print("Queries:", len(Q))
print("Evidence passages:", len(E))
print(MANIFEST.benchmark_split.value_counts())


## 4. Arabic normalization and temporal helpers — corrected implementation

Corrections in this version are mechanical/construct-alignment changes, not threshold tuning:
- correct Python regex escaping for Arabic tokenization and Gregorian-year extraction;
- NaN-safe temporal ambiguity handling;
- token/phrase-aware transition matching (no substring match on `إلى` inside unrelated words);
- explicit predecessor/successor direction for transition relations.

In [ ]:
CURRENT_CUES = [
    "حالياً","حاليا","الحالي","الحالية","هسا","هسّا","دلوقتي","الحين","الآن","الان",
    "current","currently","now","الوقت الحالي"
]

GENERIC_CUES = [
    "معلومات عامة","تنشر الجهة","الموقع الرسمي ينشر","أخباراً وتحديثات","اخبارا وتحديثات",
    "توجد مواد","توجد معلومات عامة"
]

# These cues identify an event/transition passage.  Crucially, "إلى" alone is
# NOT a transition cue.  Directional SUPERSEDES/SUPPORTS is determined only
# when a predecessor -> successor pattern can be extracted from Evidence A.
TRANSITION_EVENT_CUES = [
    "خلفاً","خلفا","بدلاً من","بدلا من","انتقل","انتقلت","تغير","تغيرت",
    "عُيّن","عين","تعيين","بدء تولي","بدأ تولي","أصبح","اصبح","ارتفعت","ارتفع",
    "دُمج","دمج","قرار دمج","اعتماد اسم","بدء تطبيق","استبدل","حل محل",
    "succeeded","replaced","appointed","merged","renamed","increased"
]

_AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")

def safe_text(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def norm_ar(s):
    s = safe_text(s).lower()
    s = _AR_DIACRITICS.sub("", s).replace("ـ", "")
    s = re.sub(r"[إأآٱا]", "ا", s)
    s = s.replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي")
    return re.sub(r"\s+", " ", s).strip()

def ar_tokens(text):
    s = norm_ar(text)
    s = re.sub(r"[^\w\u0600-\u06FF%]+", " ", s)
    return [x for x in s.split() if x]

def phrase_present(text, cue):
    t = norm_ar(text)
    c = norm_ar(cue)
    if not c:
        return False
    # Phrase/token boundaries prevent "الي" from firing inside words such as "حاليا".
    return re.search(rf"(?<!\w){re.escape(c)}(?!\w)", t) is not None

def contains_any(text, cues):
    return any(phrase_present(text, c) for c in cues)

def extract_years(text):
    return [int(x) for x in re.findall(r"(?<!\d)(19\d{2}|20\d{2})(?!\d)", safe_text(text))]

def parse_year(ref):
    s = safe_text(ref).strip()
    if not s:
        return None
    m = re.search(r"(19\d{2}|20\d{2})", s)
    return int(m.group(1)) if m else None

def is_ambiguous_query(row):
    ref = safe_text(row.get("reference_surface", ""))
    norm = safe_text(row.get("normalized_reference_time", ""))
    ref_n = norm_ar(ref)
    return (not norm.strip()) and (
        "غير محدد" in ref_n or
        "اي انتقال" in ref_n or
        "أي انتقال" in ref
    )

def is_current_query(row):
    text = safe_text(row.get("query_text", ""))
    ref = safe_text(row.get("normalized_reference_time", ""))
    return contains_any(text, CURRENT_CUES) or ref == CORPUS_REFERENCE_TIME

def is_future_query(row):
    qy = parse_year(row.get("normalized_reference_time", ""))
    cy = parse_year(CORPUS_REFERENCE_TIME)
    return qy is not None and cy is not None and qy > cy

def evidence_year_profile(text, source_date=None):
    ys = extract_years(text)
    sy = parse_year(source_date)
    if sy is not None:
        ys.append(sy)
    ys = sorted(set(ys))
    return {
        "years": ys,
        "min_year": min(ys) if ys else None,
        "max_year": max(ys) if ys else None,
    }

def temporal_fit_to_year(text, source_date, query_year):
    if query_year is None:
        return 0.0
    ys = evidence_year_profile(text, source_date)["years"]
    if not ys:
        return 0.0
    lo, hi = min(ys), max(ys)
    if lo <= query_year <= hi:
        return 1.0
    d = min(abs(y-query_year) for y in ys)
    if d == 1:
        return 0.65
    if d == 2:
        return 0.35
    return 0.0

def explicit_transition_event(text):
    return any(phrase_present(text, cue) for cue in TRANSITION_EVENT_CUES)

def transition_endpoints(text):
    """Return normalized (predecessor, successor) for explicit 'من X إلى Y' text."""
    t = norm_ar(text)
    # Add spaces so literal token boundaries are robust at string edges.
    padded = f" {t} "
    if " من " not in padded or " الي " not in padded:
        return None, None
    after_from = padded.split(" من ", 1)[1]
    if " الي " not in after_from:
        return None, None
    predecessor, successor_rest = after_from.split(" الي ", 1)
    # Generated transition passages end the successor before provenance phrases/punctuation.
    successor = re.split(r"\s+(?:وفق|بموجب|اعتبارا|ابتداء)\b|[.;؛]", successor_rest, maxsplit=1)[0]
    predecessor = predecessor.strip(" \"'«»[]()،.;؛:")
    successor = successor.strip(" \"'«»[]()،.;؛:")
    return predecessor or None, successor or None

def endpoint_in_text(endpoint, text):
    if not endpoint:
        return False
    e = norm_ar(endpoint).strip(" \"'«»[]()،.;؛:")
    t = norm_ar(text)
    return bool(e) and e in t

def transition_direction(source_text, target_text):
    """Round-3 semantics: transition->successor SUPPORTS; transition->predecessor SUPERSEDES."""
    old, new = transition_endpoints(source_text)
    if old and endpoint_in_text(old, target_text):
        return "SUPERSEDES"
    if new and endpoint_in_text(new, target_text):
        return "SUPPORTS"
    return None

print("Corrected helpers ready.")
print("Sanity years:", extract_years("في 2019 ثم 2024-09-18"))
print("Sanity Arabic tokens:", ar_tokens("من هو الوزير الحالي في الأردن؟")[:8])


## 5. Load frozen dense retriever and multilingual NLI model


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

dense_model = SentenceTransformer(DENSE_MODEL_ID, device=DEVICE)

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_ID).to(DEVICE)
nli_model.eval()

id2label = {int(k):str(v).lower() for k,v in nli_model.config.id2label.items()}
print("NLI labels:", id2label)

def find_label_idx(fragment):
    for idx,label in id2label.items():
        if fragment in label:
            return idx
    raise ValueError(f"Cannot find {fragment} in {id2label}")

ENTAIL_IDX = find_label_idx("entail")
CONTRA_IDX = find_label_idx("contra")
NEUTRAL_IDX = find_label_idx("neutral")

RUN_ENV["dense_model_commit"] = getattr(dense_model[0].auto_model.config, "_commit_hash", None)
RUN_ENV["nli_model_commit"] = getattr(nli_model.config, "_commit_hash", None)
print(json.dumps(RUN_ENV, indent=2))


## 6. Encode queries and evidence; compute BM25 and E5 retrieval scores


In [ ]:
q_texts = ["query: " + x for x in Q.query_text.astype(str)]
e_texts = ["passage: " + x for x in E.evidence_text.astype(str)]

Q_EMB = dense_model.encode(
    q_texts, normalize_embeddings=True, convert_to_numpy=True,
    show_progress_bar=True, batch_size=64
)
E_EMB = dense_model.encode(
    e_texts, normalize_embeddings=True, convert_to_numpy=True,
    show_progress_bar=True, batch_size=64
)

qid_to_emb = dict(zip(Q.query_id, Q_EMB))
eid_to_emb = dict(zip(E.evidence_id, E_EMB))
eid_to_text = E.set_index("evidence_id").evidence_text.to_dict()
eid_to_source_date = E.set_index("evidence_id").source_date.to_dict()

def minmax(x):
    x=np.asarray(x,dtype=float)
    if len(x)==0: return x
    if float(x.max())==float(x.min()): return np.zeros_like(x)
    return (x-x.min())/(x.max()-x.min())

retrieval_rows=[]
for _,qr in Q.iterrows():
    pack=E[E.scenario_id==qr.scenario_id].copy().reset_index(drop=True)

    bm=BM25Okapi([ar_tokens(x) for x in pack.evidence_text.astype(str)])
    bm_raw=np.asarray(bm.get_scores(ar_tokens(qr.query_text)),dtype=float)
    bm_norm=minmax(bm_raw)

    qv=qid_to_emb[qr.query_id]
    dense_raw=np.array([float(np.dot(qv,eid_to_emb[eid])) for eid in pack.evidence_id])
    dense_norm=minmax(dense_raw)

    for i,er in pack.iterrows():
        retrieval_rows.append({
            "query_id":qr.query_id,
            "scenario_id":qr.scenario_id,
            "evidence_id":er.evidence_id,
            "bm25_raw":float(bm_raw[i]),
            "bm25_semantic":float(bm_norm[i]),
            "e5_raw":float(dense_raw[i]),
            "e5_semantic":float(dense_norm[i]),
        })

RETRIEVAL=pd.DataFrame(retrieval_rows)
print(RETRIEVAL.head())


## 7. Batched NLI scoring


In [ ]:
@torch.inference_mode()
def nli_batch(pairs, batch_size=32):
    # pairs: list[(premise, hypothesis)]
    out=[]
    for start in range(0,len(pairs),batch_size):
        batch=pairs[start:start+batch_size]
        enc=nli_tokenizer(
            [a for a,b in batch],
            [b for a,b in batch],
            padding=True,truncation=True,max_length=512,
            return_tensors="pt"
        ).to(DEVICE)
        logits=nli_model(**enc).logits.detach().cpu().numpy()
        probs=softmax(logits,axis=1)
        for p in probs:
            out.append({
                "entailment":float(p[ENTAIL_IDX]),
                "contradiction":float(p[CONTRA_IDX]),
                "neutral":float(p[NEUTRAL_IDX]),
            })
    return out


## 8. Frozen Pairwise Evidence–Evidence Relation Reasoner — Round-3 aligned

The relation construct is now evaluated **once per scenario**, independently of the query:

\[
R_{ij}=g(e_i,e_j)
\]

This directly matches the construct validated in Relation Calibration Round 3. Query time and query relevance are used only in QCES inference, not to define the pairwise relation.

Operational direction is explicit:
- transition `X → Y` to a passage asserting `Y` → `SUPPORTS`;
- transition `X → Y` to a passage asserting `X` → `SUPERSEDES`;
- incompatible same-time claims → `CONTRADICTS`;
- otherwise → `NONE`.

A merely newer state passage is **never** enough for `SUPERSEDES`.

In [ ]:
# Cache pairwise E5 similarities.
def evidence_cos(eid_a, eid_b):
    return float(np.dot(eid_to_emb[eid_a], eid_to_emb[eid_b]))

def same_time_text_only(a_text, b_text):
    # Both explicitly framed as current.
    if contains_any(a_text, CURRENT_CUES) and contains_any(b_text, CURRENT_CUES):
        return True

    # Or both explicitly anchor at least one identical Gregorian year.
    ya = set(extract_years(a_text))
    yb = set(extract_years(b_text))
    if ya & yb:
        return True

    return False

def relation_rule(eid_a, eid_b, nli):
    a = eid_to_text[eid_a]
    b = eid_to_text[eid_b]
    pair_sem = evidence_cos(eid_a, eid_b)

    # 1) Co-temporal incompatible claims. Query relevance is intentionally absent here;
    # relevance is applied later by QCES.
    if (
        same_time_text_only(a, b)
        and pair_sem >= SEMANTIC_RELATION_FLOOR
        and nli["contradiction"] >= NLI_CONTRADICTION_THRESHOLD
    ):
        return "CONTRADICTS", nli["contradiction"], "same_time_text_nli_conflict"

    # 2) Explicit directional transition/replacement.
    direction = transition_direction(a, b)
    if direction is not None and pair_sem >= SEMANTIC_RELATION_FLOOR:
        base = 0.62 if direction == "SUPERSEDES" else 0.60
        conf = max(base, min(0.97, base + 0.22 * pair_sem))
        return direction, float(conf), "explicit_transition_direction"

    # 3) General textual entailment/corroboration.
    if (
        nli["entailment"] >= NLI_ENTAILMENT_THRESHOLD
        and pair_sem >= SEMANTIC_RELATION_FLOOR
    ):
        return "SUPPORTS", nli["entailment"], "nli_entailment"

    return "NONE", max(nli["neutral"], 1 - pair_sem), "fallback_none"

# Build all ordered evidence pairs ONCE per scenario (query-independent R_ij).
pair_jobs = []
for sid, pack in E.groupby("scenario_id"):
    eids = pack.evidence_id.tolist()
    for ea in eids:
        for eb in eids:
            if ea == eb:
                continue
            pair_jobs.append((sid, ea, eb))

print("Ordered scenario-level pair decisions to score:", len(pair_jobs))

nli_inputs = [(eid_to_text[ea], eid_to_text[eb]) for sid, ea, eb in pair_jobs]
nli_outputs = nli_batch(nli_inputs, batch_size=32)

REL_ROWS = []
for (sid, ea, eb), nli in zip(pair_jobs, nli_outputs):
    label, conf, reason = relation_rule(ea, eb, nli)
    old_value, new_value = transition_endpoints(eid_to_text[ea])
    REL_ROWS.append({
        "scenario_id": sid,
        "evidence_A": ea,
        "evidence_B": eb,
        "pred_relation": label,
        "relation_confidence": conf,
        "relation_reason": reason,
        "nli_entailment": nli["entailment"],
        "nli_contradiction": nli["contradiction"],
        "nli_neutral": nli["neutral"],
        "pair_e5_cosine": evidence_cos(ea, eb),
        "transition_predecessor": old_value,
        "transition_successor": new_value,
    })

REL_SCENARIO = pd.DataFrame(REL_ROWS)
print(REL_SCENARIO.pred_relation.value_counts())

# Mechanical sanity checks that do not use gold labels.
assert not REL_SCENARIO.duplicated(["scenario_id", "evidence_A", "evidence_B"]).any()
assert len(REL_SCENARIO) == sum(len(g) * (len(g)-1) for _, g in E.groupby("scenario_id"))


## 9. Query-conditioned QCES inference


In [ ]:
def relation_lookup(scenario_id):
    g = REL_SCENARIO[REL_SCENARIO.scenario_id == scenario_id]
    incoming = defaultdict(list)
    outgoing = defaultdict(list)
    for _, r in g.iterrows():
        outgoing[r.evidence_A].append(r)
        incoming[r.evidence_B].append(r)
    return incoming, outgoing

def infer_states_for_query(qrow, semantic_col="e5_semantic"):
    pack = E[E.scenario_id == qrow.scenario_id]
    ret = RETRIEVAL[RETRIEVAL.query_id == qrow.query_id].set_index("evidence_id")
    incoming, outgoing = relation_lookup(qrow.scenario_id)

    ambiguous = is_ambiguous_query(qrow)
    future = is_future_query(qrow)
    current = is_current_query(qrow)
    qyear = parse_year(qrow.get("normalized_reference_time", ""))

    # First pass: evidence is defeated only by an explicit pairwise SUPERSEDES edge
    # from a sufficiently query-relevant source passage.
    defeated = set()
    defeat_conf = {}
    for eid in pack.evidence_id:
        for r in incoming.get(eid, []):
            if r.pred_relation != "SUPERSEDES":
                continue
            src = r.evidence_A
            src_rel = float(ret.loc[src, semantic_col])
            if src_rel < SEMANTIC_RELATION_FLOOR:
                continue

            # Historical protection: a transition after the requested year must not
            # defeat evidence that is valid for that earlier historical year.
            if qyear is not None and not current:
                src_prof = evidence_year_profile(
                    eid_to_text[src], eid_to_source_date.get(src, "")
                )
                sy = src_prof["max_year"]
                if sy is not None and sy > qyear:
                    continue

            defeated.add(eid)
            defeat_conf[eid] = max(
                defeat_conf.get(eid, 0.0), float(r.relation_confidence)
            )

    states = []
    for _, er in pack.iterrows():
        eid = er.evidence_id
        sem = float(ret.loc[eid, semantic_col])
        text = er.evidence_text
        generic = contains_any(text, GENERIC_CUES)

        if ambiguous or future:
            state = "INSUFFICIENT"
            conf = 0.95
        elif eid in defeated:
            state = "SUPERSEDED"
            conf = max(0.60, defeat_conf.get(eid, 0.60))
        else:
            # Query-conditioned conflict propagation. Pairwise CONTRADICTS exists
            # independently; it becomes CO_TEMPORAL_CONFLICT only when both sides
            # are sufficiently relevant to this query and neither has been defeated.
            conflict_conf = 0.0
            for r in incoming.get(eid, []) + outgoing.get(eid, []):
                if r.pred_relation != "CONTRADICTS":
                    continue
                other = r.evidence_A if r.evidence_B == eid else r.evidence_B
                if other in defeated:
                    continue
                other_sem = float(ret.loc[other, semantic_col])
                if (
                    sem >= CURRENT_CONFLICT_SUPPORT_FLOOR
                    and other_sem >= CURRENT_CONFLICT_SUPPORT_FLOOR
                ):
                    conflict_conf = max(
                        conflict_conf, float(r.relation_confidence)
                    )

            if conflict_conf > 0:
                state = "CO_TEMPORAL_CONFLICT"
                conf = max(0.60, conflict_conf)
            elif generic or sem < 0.20:
                state = "INSUFFICIENT"
                conf = max(0.55, 1 - sem)
            elif current:
                state = "ACTIVE_SUPPORT"
                conf = max(0.55, sem)
            else:
                fit = temporal_fit_to_year(text, er.source_date, qyear)
                if fit >= 0.65 and sem >= 0.25:
                    state = "HISTORICAL_SUPPORT"
                    conf = max(0.55, 0.5 * sem + 0.5 * fit)
                else:
                    state = "INSUFFICIENT"
                    conf = max(0.50, 1 - fit)

        states.append({
            "query_id": qrow.query_id,
            "scenario_id": qrow.scenario_id,
            "evidence_id": eid,
            "predicted_state": state,
            "state_confidence": float(min(0.99, conf)),
            "semantic_score": sem,
        })
    return pd.DataFrame(states)

STATE_E5 = pd.concat(
    [infer_states_for_query(qr, "e5_semantic") for _, qr in Q.iterrows()],
    ignore_index=True,
)
STATE_BM25 = pd.concat(
    [infer_states_for_query(qr, "bm25_semantic") for _, qr in Q.iterrows()],
    ignore_index=True,
)

print("E5 states:")
print(STATE_E5.predicted_state.value_counts())


## 10. Frozen graph-aware reranking and action policy


In [ ]:
def graph_bonus_for_scenario(scenario_id):
    g = REL_SCENARIO[REL_SCENARIO.scenario_id == scenario_id]
    eids = E[E.scenario_id == scenario_id].evidence_id.tolist()
    out = {}
    for eid in eids:
        support = 0.0
        superseded_in = 0.0
        conflict = 0.0
        for _, r in g.iterrows():
            if r.evidence_B == eid and r.pred_relation == "SUPPORTS":
                support += r.relation_confidence
            if r.evidence_B == eid and r.pred_relation == "SUPERSEDES":
                superseded_in += r.relation_confidence
            if (
                (r.evidence_A == eid or r.evidence_B == eid)
                and r.pred_relation == "CONTRADICTS"
            ):
                conflict += 0.5 * r.relation_confidence
        raw = 0.5 * support - superseded_in - 0.5 * conflict
        out[eid] = float(np.clip(raw, -1, 1))
    return out

GRAPH_BONUS = {
    sid: graph_bonus_for_scenario(sid) for sid in E.scenario_id.unique()
}

def rerank(states, semantic_col_name, system_name):
    rows = []
    for qid, g in states.groupby("query_id"):
        gg = g.copy()
        scenario_id = gg.scenario_id.iloc[0]
        gg["state_bonus"] = gg.predicted_state.map(STATE_BONUS).fillna(0.0)
        gg["graph_bonus"] = gg.evidence_id.map(GRAPH_BONUS[scenario_id]).fillna(0.0)
        gg["final_score"] = (
            SEMANTIC_WEIGHT * gg.semantic_score
            + STATE_WEIGHT * gg.state_bonus * gg.state_confidence
            + GRAPH_WEIGHT * gg.graph_bonus
        )
        best = gg.sort_values(
            ["final_score", "semantic_score"], ascending=False
        ).iloc[0]
        rows.append({
            "system": system_name,
            "query_id": qid,
            "scenario_id": best.scenario_id,
            "selected_evidence_id": best.evidence_id,
            "selected_state": best.predicted_state,
            "selected_score": best.final_score,
        })
    return pd.DataFrame(rows)

PRED_E5 = rerank(STATE_E5, "e5_semantic", "Dense-E5+AraVISTA")
PRED_BM25 = rerank(STATE_BM25, "bm25_semantic", "BM25+AraVISTA")

def action_for_query(qid, qrow, state_df, selected_eid):
    sub = state_df[state_df["query_id"] == qid]

    if is_ambiguous_query(qrow):
        return "CLARIFY"
    if is_future_query(qrow):
        return "ABSTAIN_INSUFFICIENT"

    if (sub["predicted_state"] == "CO_TEMPORAL_CONFLICT").sum() >= 2:
        return "ABSTAIN_CONFLICT"

    supportive = sub[sub["predicted_state"].isin(
        ["ACTIVE_SUPPORT", "HISTORICAL_SUPPORT"]
    )]
    if supportive.empty:
        return "ABSTAIN_INSUFFICIENT"

    if is_current_query(qrow) and (sub["predicted_state"] == "SUPERSEDED").any():
        return "ANSWER_WITH_UPDATE"

    return "ANSWER"

def attach_actions(pred, state_df):
    rows = []
    qidx = Q.set_index("query_id")
    for _, r in pred.iterrows():
        qid = r["query_id"]
        qr = qidx.loc[qid]
        rr = r.to_dict()
        rr["predicted_action"] = action_for_query(
            qid=qid,
            qrow=qr,
            state_df=state_df,
            selected_eid=r["selected_evidence_id"],
        )
        rows.append(rr)
    return pd.DataFrame(rows)

PRED_E5 = attach_actions(PRED_E5, STATE_E5)
PRED_BM25 = attach_actions(PRED_BM25, STATE_BM25)

display(PRED_E5.head())
print("Predicted action distribution (E5):")
print(PRED_E5.predicted_action.value_counts())


## Optional checkpoint before gold upload

On free Colab, you may save the expensive inference outputs **before** uploading any gold. This does not affect predictions. The cell below writes only leakage-safe predictions to `/content/AraVISTA_checkpoints`.

In [ ]:
CHECKPOINT_DIR = "/content/AraVISTA_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
REL_SCENARIO.to_csv(f"{CHECKPOINT_DIR}/REL_SCENARIO.csv", index=False, encoding="utf-8-sig")
STATE_E5.to_csv(f"{CHECKPOINT_DIR}/STATE_E5.csv", index=False, encoding="utf-8-sig")
STATE_BM25.to_csv(f"{CHECKPOINT_DIR}/STATE_BM25.csv", index=False, encoding="utf-8-sig")
PRED_E5.to_csv(f"{CHECKPOINT_DIR}/PRED_E5.csv", index=False, encoding="utf-8-sig")
PRED_BM25.to_csv(f"{CHECKPOINT_DIR}/PRED_BM25.csv", index=False, encoding="utf-8-sig")
print("Leakage-safe checkpoint written to", CHECKPOINT_DIR)


## HARD TEST LOCK

The ontology gates have passed. The lock remains explicit so the notebook preserves the original protocol boundary: all predictions above are computed before any gold file is uploaded.

In [ ]:
if not ALLOW_TEST_EVALUATION:
    raise RuntimeError(
        "TEST LOCK ACTIVE. Keep ALLOW_TEST_EVALUATION=False until both "
        "Calibration Round 2 Cohen-kappa gates are >= 0.75."
    )
print("TEST LOCK OPENED. Proceeding to one locked evaluation.")


## 12. Evaluation-only upload

Only after inference above has completed, upload:

- `benchmark_240_v1.1.csv`
- `evidence_packs_v1.1.csv`
- `qces_gold_v1.1.csv`
- `relation_graph_v1.1_R3_aligned.csv`

The relation file is the **Round-3 ontology-aligned gold revision**. Query, evidence, QCES, answer, and action gold are unchanged from AraVISTA-Bench v1.1.

In [ ]:
gold_upload = files.upload()

GQ = pd.read_csv("benchmark_240_v1.1.csv")
GE = pd.read_csv("evidence_packs_v1.1.csv")
GSTATE = pd.read_csv("qces_gold_v1.1.csv")
GREL = pd.read_csv(RELATION_GOLD_FILE)

candidate_answer = GE.set_index("evidence_id").candidate_answer.to_dict()

# Freeze prediction snapshots immediately after gold is loaded.
PRED_E5_FROZEN = PRED_E5.copy(deep=True)
PRED_BM25_FROZEN = PRED_BM25.copy(deep=True)
REL_SCENARIO_FROZEN = REL_SCENARIO.copy(deep=True)
STATE_E5_FROZEN = STATE_E5.copy(deep=True)
STATE_BM25_FROZEN = STATE_BM25.copy(deep=True)

print("Evaluation gold loaded after inference.")
print("Relation gold:", RELATION_GOLD_FILE)


## 13. Relation and QCES evaluation


In [ ]:
# Pairwise relation evaluation: benchmark-selected scenario-level edges only.
rel_pred = REL_SCENARIO_FROZEN.merge(
    GREL[["scenario_id","source_evidence_id","target_evidence_id","gold_relation"]],
    left_on=["scenario_id","evidence_A","evidence_B"],
    right_on=["scenario_id","source_evidence_id","target_evidence_id"],
    how="inner"
)

print("Matched relation edges:", len(rel_pred), "/", len(GREL))
assert len(rel_pred) == len(GREL), "Every ontology-aligned gold edge must match one predicted scenario-level pair."
REL_ACC = accuracy_score(rel_pred.gold_relation, rel_pred.pred_relation)
REL_F1 = f1_score(rel_pred.gold_relation, rel_pred.pred_relation, average="macro")
print("Relation Accuracy:", REL_ACC)
print("Relation Macro-F1:", REL_F1)
print(classification_report(rel_pred.gold_relation, rel_pred.pred_relation, digits=3))

def state_eval(pred_states, label):
    x = pred_states.merge(
        GSTATE[["query_id","evidence_id","qces_state"]],
        on=["query_id","evidence_id"], how="inner"
    )
    acc = accuracy_score(x.qces_state, x.predicted_state)
    mf1 = f1_score(x.qces_state, x.predicted_state, average="macro")
    print("\n", label, "QCES accuracy", acc, "Macro-F1", mf1)
    print(classification_report(x.qces_state, x.predicted_state, digits=3))
    return x, acc, mf1

STATE_E5_EVAL, STATE_E5_ACC, STATE_E5_F1 = state_eval(STATE_E5_FROZEN, "E5")
STATE_BM25_EVAL, STATE_BM25_ACC, STATE_BM25_F1 = state_eval(STATE_BM25_FROZEN, "BM25")


## 14. End-to-end evaluation with multi-gold evidence support


In [ ]:
GQ_IDX=GQ.set_index("query_id")

def evaluate_predictions(pred,label):
    x=pred.copy()
    x=x.merge(
        GQ[[
            "query_id","scenario_id","family","benchmark_split","gold_answer",
            "gold_action","gold_evidence_ids","pair_group_id"
        ]],
        on=["query_id","scenario_id"],how="left"
    )
    x["pred_answer"]=x.selected_evidence_id.map(candidate_answer).fillna("")
    x["gold_evidence_set"]=x.gold_evidence_ids.fillna("").apply(
        lambda s:set(v for v in str(s).split(";") if v)
    )
    x["evidence_correct"]=x.apply(
        lambda r:(r.selected_evidence_id in r.gold_evidence_set) if r.gold_evidence_set else np.nan,
        axis=1
    )
    answerable=x.gold_action.isin(["ANSWER","ANSWER_WITH_UPDATE"])
    x["answer_correct"]=np.where(
        answerable,
        x.pred_answer.astype(str)==x.gold_answer.fillna("").astype(str),
        np.nan
    )
    x["action_correct"]=x.predicted_action.eq(x.gold_action)

    out={
        "System":label,
        "Action_Accuracy":float(x.action_correct.mean()),
        "Evidence_Top1_Accuracy":float(x.evidence_correct.dropna().mean()),
        "Answer_Accuracy_answerable":float(pd.Series(x.answer_correct).dropna().mean()),
    }

    # Core / calendar separation.
    core=x[x.benchmark_split=="HELDOUT_NEW_CHAIN"]
    cal=x[x.benchmark_split=="CALENDAR_CHALLENGE"]
    out["Core_Action_Accuracy"]=float(core.action_correct.mean())
    out["Core_Answer_Accuracy_answerable"]=float(pd.Series(core.answer_correct).dropna().mean())
    out["Calendar_Action_Accuracy"]=float(cal.action_correct.mean())

    # Family accuracies.
    for fam,g in x.groupby("family"):
        out[f"{fam}_Action_Accuracy"]=float(g.action_correct.mean())
        aa=pd.Series(g.answer_correct).dropna()
        out[f"{fam}_Answer_Accuracy"]=float(aa.mean()) if len(aa) else np.nan

    # Temporal twins.
    sens=[]; twins=[]
    for pg,g in x[x.family=="TEMPORAL_COUNTERFACTUAL"].groupby("pair_group_id"):
        vals=g.pred_answer.astype(str).tolist()
        if len(vals)==2:
            sens.append(vals[0]!=vals[1])
            twins.append(bool(pd.Series(g.answer_correct).fillna(False).all()))
    out["Temporal_Sensitivity"]=float(np.mean(sens))
    out["Temporal_Twin_Correctness"]=float(np.mean(twins))

    # Invariance.
    for fam,prefix in [
        ("CALENDAR_EQUIVALENT","Calendar"),
        ("DIALECT_TEMPORAL_EQUIVALENT","Dialect")
    ]:
        inv=[]; cinv=[]
        for pg,g in x[x.family==fam].groupby("pair_group_id"):
            vals=g.pred_answer.astype(str).tolist()
            same=(len(set(vals))==1)
            inv.append(same)
            corr=pd.Series(g.answer_correct).fillna(False).all()
            cinv.append(bool(same and corr))
        out[f"{prefix}_Invariance"]=float(np.mean(inv)) if inv else np.nan
        out[f"Correct_{prefix}_Invariance"]=float(np.mean(cinv)) if cinv else np.nan

    st=x[x.family=="STALE_EVIDENCE_TRAP"]
    out["Stale_Trap_Evidence_Success"]=float(st.evidence_correct.dropna().mean())

    ctrl=x[x.family=="CONTROL_OR_INSUFFICIENT"]
    out["Control_Action_Accuracy"]=float(ctrl.action_correct.mean())

    return x,out

EVAL_E5,SUM_E5=evaluate_predictions(PRED_E5_FROZEN,"Dense-E5+AraVISTA")
EVAL_BM25,SUM_BM25=evaluate_predictions(PRED_BM25_FROZEN,"BM25+AraVISTA")

SUMMARY=pd.DataFrame([SUM_E5,SUM_BM25])
display(SUMMARY.T)


## 15. Retrieval-only baselines


In [ ]:
def retrieval_top1(score_col,system):
    rows=[]
    for qid,g in RETRIEVAL.groupby("query_id"):
        best=g.sort_values(score_col,ascending=False).iloc[0]
        rows.append({
            "system":system,
            "query_id":qid,
            "scenario_id":best.scenario_id,
            "selected_evidence_id":best.evidence_id,
            "predicted_action":"ANSWER",
        })
    return pd.DataFrame(rows)

BASE_BM25=retrieval_top1("bm25_semantic","BM25")
BASE_E5=retrieval_top1("e5_semantic","Dense-E5")

BASE_E5_EVAL,BASE_E5_SUM=evaluate_predictions(BASE_E5,"Dense-E5")
BASE_BM25_EVAL,BASE_BM25_SUM=evaluate_predictions(BASE_BM25,"BM25")

ALL_SUMMARY=pd.DataFrame([BASE_BM25_SUM,BASE_E5_SUM,SUM_BM25,SUM_E5])
display(ALL_SUMMARY.T)


## 16. Scenario-level bootstrap confidence intervals


In [ ]:
def scenario_bootstrap_metric(eval_df, metric_fn, n=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED):
    rng=np.random.default_rng(seed)
    sids=eval_df.scenario_id.unique()
    vals=[]
    for _ in range(n):
        sampled=rng.choice(sids,size=len(sids),replace=True)
        parts=[]
        for j,sid in enumerate(sampled):
            g=eval_df[eval_df.scenario_id==sid].copy()
            # unique bootstrap cluster ID preserves repeated sampled scenarios.
            g["_boot_cluster"]=f"{sid}__{j}"
            parts.append(g)
        b=pd.concat(parts,ignore_index=True)
        vals.append(metric_fn(b))
    vals=np.asarray(vals,dtype=float)
    return np.nanpercentile(vals,[2.5,50,97.5])

def action_acc(df): return float(df.action_correct.mean())
def answer_acc(df): return float(pd.Series(df.answer_correct).dropna().mean())
def evidence_acc(df): return float(df.evidence_correct.dropna().mean())

CI_ROWS=[]
for name,df in [
    ("Dense-E5",BASE_E5_EVAL),
    ("BM25",BASE_BM25_EVAL),
    ("Dense-E5+AraVISTA",EVAL_E5),
    ("BM25+AraVISTA",EVAL_BM25),
]:
    core=df[df.benchmark_split=="HELDOUT_NEW_CHAIN"]
    for metric_name,fn in [
        ("Core_Action_Accuracy",action_acc),
        ("Core_Answer_Accuracy",answer_acc),
        ("Core_Evidence_Accuracy",evidence_acc),
    ]:
        lo,mid,hi=scenario_bootstrap_metric(core,fn)
        CI_ROWS.append({
            "System":name,"Metric":metric_name,
            "CI_2.5":lo,"Bootstrap_Median":mid,"CI_97.5":hi
        })

CI_TABLE=pd.DataFrame(CI_ROWS)
display(CI_TABLE)


## 17. Export immutable corrected-run artifacts

This cell writes **versioned** outputs so the original diagnostic run remains untouched. It also creates one ZIP file for easy upload back to ChatGPT.

In [ ]:
RUN_MANIFEST = {
    **RUN_ENV,
    "run_version": RUN_VERSION,
    "implementation_status": "Round-3 ontology aligned; regex/NaN/transition-direction bugs corrected",
    "relation_gold_file": RELATION_GOLD_FILE,
    "pairwise_relation_definition": "R_ij = g(e_i,e_j), query-independent",
    "corpus_reference_time": CORPUS_REFERENCE_TIME,
    "thresholds": {
        "nli_contradiction": NLI_CONTRADICTION_THRESHOLD,
        "nli_entailment": NLI_ENTAILMENT_THRESHOLD,
        "semantic_relation_floor": SEMANTIC_RELATION_FLOOR,
        "current_conflict_support_floor": CURRENT_CONFLICT_SUPPORT_FLOOR,
    },
    "weights": {
        "semantic": SEMANTIC_WEIGHT,
        "state": STATE_WEIGHT,
        "graph": GRAPH_WEIGHT,
    },
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "random_seed": RANDOM_SEED,
}

PREFIX = "aravista_frozen_v1_1"
OUTPUT_FILES = [
    f"{PREFIX}_pairwise_predictions.csv",
    f"{PREFIX}_qces_e5.csv",
    f"{PREFIX}_qces_bm25.csv",
    f"{PREFIX}_end2end_e5.csv",
    f"{PREFIX}_end2end_bm25.csv",
    f"{PREFIX}_summary.csv",
    f"{PREFIX}_bootstrap_ci.csv",
    f"{PREFIX}_run_manifest.json",
]

REL_SCENARIO_FROZEN.to_csv(OUTPUT_FILES[0], index=False, encoding="utf-8-sig")
STATE_E5_FROZEN.to_csv(OUTPUT_FILES[1], index=False, encoding="utf-8-sig")
STATE_BM25_FROZEN.to_csv(OUTPUT_FILES[2], index=False, encoding="utf-8-sig")
EVAL_E5.to_csv(OUTPUT_FILES[3], index=False, encoding="utf-8-sig")
EVAL_BM25.to_csv(OUTPUT_FILES[4], index=False, encoding="utf-8-sig")
ALL_SUMMARY.to_csv(OUTPUT_FILES[5], index=False, encoding="utf-8-sig")
CI_TABLE.to_csv(OUTPUT_FILES[6], index=False, encoding="utf-8-sig")

with open(OUTPUT_FILES[7], "w", encoding="utf-8") as f:
    json.dump(RUN_MANIFEST, f, indent=2, ensure_ascii=False)

import zipfile
ZIP_NAME = "AraVISTA_Frozen_Run_v1.1_Results.zip"
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in OUTPUT_FILES:
        z.write(fn, arcname=fn)

from google.colab import files
files.download(ZIP_NAME)

print("Corrected locked evaluation export complete.")
print("Created:", ZIP_NAME)
